# Taller BI - Juegos Olimpicos

Notebook del ETL en Python solicitado en el taller. Carga los CSV, limpia datos,
construye un modelo estrella, crea indicadores, almacena el resultado en SQLite
y genera visualizaciones con Plotly.

In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd
import plotly.express as px

SOURCE_DIR = Path(r"C:\Users\david\OneDrive\Documentos\BIG DATA")
OUT_DIR = SOURCE_DIR / "entregables_taller_bi"
CSV_DIR = OUT_DIR / "csv_powerbi"
OUT_DIR.mkdir(exist_ok=True)
CSV_DIR.mkdir(exist_ok=True)

## 1. Extraccion

Se leen los cinco archivos planos del proyecto.

In [ ]:
athletes = pd.read_csv(SOURCE_DIR / "Athletes_Games.csv")
editions = pd.read_csv(SOURCE_DIR / "Edition_Games.csv")
teams = pd.read_csv(SOURCE_DIR / "equipos_Games.csv")
events = pd.read_csv(SOURCE_DIR / "Events_Olimpycs.csv")
fact_src = pd.read_csv(SOURCE_DIR / "JJ.OO.csv")
{name: df.shape for name, df in {
    "athletes": athletes, "editions": editions, "teams": teams,
    "events": events, "fact": fact_src
}.items()}

## 2. Perfilamiento y calidad

In [ ]:
for name, df in {"athletes": athletes, "editions": editions, "teams": teams, "events": events, "fact": fact_src}.items():
    print(name, "shape:", df.shape, "duplicados:", df.duplicated().sum(), "nulos:", df.isna().sum().sum())
print("Competencias distintas:", events[["Competition_id", "Sport", "Event"]].drop_duplicates().shape[0])
print(fact_src["Medal"].value_counts())

## 3. Transformacion

Se construyen dimensiones, tabla de hechos e indicadores.

In [ ]:
def clean_text_columns(df):
    df = df.copy()
    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].astype(str).str.strip()
    return df

def infer_event_gender(event_name):
    text = str(event_name)
    if "Women's" in text or "Women" in text:
        return "Femenino"
    if "Men's" in text or "Men" in text:
        return "Masculino"
    if "Mixed" in text or "Open" in text:
        return "Mixto/Open"
    return "No especificado"

athletes, editions, teams, events, fact_src = [clean_text_columns(df) for df in [athletes, editions, teams, events, fact_src]]

dim_atleta = athletes.rename(columns={"athlete_id":"atleta_id", "ID":"id_original_atleta", "Year":"anio_registro", "Name":"nombre_atleta", "Sex":"sexo", "Age":"edad", "Height":"altura_cm", "Weight":"peso_kg"})[["atleta_id", "id_original_atleta", "anio_registro", "nombre_atleta", "sexo", "edad", "altura_cm", "peso_kg"]]
dim_equipo = teams.rename(columns={"team_id":"equipo_id", "Team":"equipo", "NOC":"noc"})[["equipo_id", "equipo", "noc"]]
dim_edicion = editions.rename(columns={"edition_id":"edicion_id", "Games":"juegos", "Year":"anio", "Season":"temporada", "City":"ciudad"})[["edicion_id", "juegos", "anio", "temporada", "ciudad"]]
dim_competencia = events[["Competition_id", "Sport", "Event"]].drop_duplicates().rename(columns={"Competition_id":"competencia_id", "Sport":"deporte", "Event":"evento"}).sort_values("competencia_id")
dim_competencia["genero_evento"] = dim_competencia["evento"].map(infer_event_gender)

fact_participacion = fact_src.rename(columns={"id_participacion":"participacion_id", "Medal":"medalla", "athlete_id":"atleta_id", "team_id":"equipo_id", "edition_id":"edicion_id", "Competition_id":"competencia_id"})[["participacion_id", "medalla", "atleta_id", "equipo_id", "edicion_id", "competencia_id"]].copy()
fact_participacion["edad"] = pd.to_numeric(events["Age"], errors="coerce")
fact_participacion["altura_cm"] = pd.to_numeric(events["Height"], errors="coerce")
fact_participacion["peso_kg"] = pd.to_numeric(events["Weight"], errors="coerce")
fact_participacion["obtuvo_medalla"] = (fact_participacion["medalla"] != "No Medal").astype(int)
fact_participacion["oro"] = (fact_participacion["medalla"] == "Gold").astype(int)
fact_participacion["plata"] = (fact_participacion["medalla"] == "Silver").astype(int)
fact_participacion["bronce"] = (fact_participacion["medalla"] == "Bronze").astype(int)
fact_participacion["puntaje_medalla"] = fact_participacion["medalla"].map({"No Medal":0, "Bronze":1, "Silver":2, "Gold":3}).astype(int)
fact_participacion.head()

## 4. Carga a SQLite y CSV para Power BI

In [ ]:
tables = {"dim_atleta": dim_atleta, "dim_equipo": dim_equipo, "dim_edicion": dim_edicion, "dim_competencia": dim_competencia, "fact_participacion": fact_participacion}
for name, df in tables.items():
    df.to_csv(CSV_DIR / f"{name}.csv", index=False, encoding="utf-8-sig")

sqlite_path = OUT_DIR / "olimpicos_bi.sqlite"
if sqlite_path.exists():
    sqlite_path.unlink()
with sqlite3.connect(sqlite_path) as conn:
    for name, df in tables.items():
        df.to_sql(name, conn, index=False, if_exists="replace")
print(sqlite_path)

## 5. Indicadores

In [ ]:
analitica = fact_participacion.merge(dim_equipo, on="equipo_id").merge(dim_edicion, on="edicion_id").merge(dim_competencia, on="competencia_id")
indicadores = {
    "participaciones": len(fact_participacion),
    "medallas": int(fact_participacion["obtuvo_medalla"].sum()),
    "oro": int(fact_participacion["oro"].sum()),
    "plata": int(fact_participacion["plata"].sum()),
    "bronce": int(fact_participacion["bronce"].sum()),
    "tasa_medalla": fact_participacion["obtuvo_medalla"].mean(),
}
indicadores

## 6. Visualizaciones con Plotly

In [ ]:
medallas = analitica[analitica["obtuvo_medalla"] == 1]

top_noc = medallas.groupby(["noc", "medalla"], as_index=False).size().rename(columns={"size":"conteo"})
top_noc_names = medallas.groupby("noc").size().sort_values(ascending=False).head(15).index
px.bar(top_noc[top_noc["noc"].isin(top_noc_names)], x="conteo", y="noc", color="medalla", orientation="h", title="Top NOC por medallas").show()

trend = medallas.groupby(["anio", "temporada"], as_index=False).size().rename(columns={"size":"medallas"})
px.line(trend, x="anio", y="medallas", color="temporada", markers=True, title="Medallas por anio").show()

sports = medallas.groupby(["deporte", "medalla"], as_index=False).size().rename(columns={"size":"conteo"})
top_sports = medallas.groupby("deporte").size().sort_values(ascending=False).head(12).index
px.bar(sports[sports["deporte"].isin(top_sports)], x="deporte", y="conteo", color="medalla", title="Medallas por deporte").show()

sex_year = analitica.merge(dim_atleta[["atleta_id", "sexo"]], on="atleta_id").groupby(["anio", "sexo"], as_index=False).size().rename(columns={"size":"participaciones"})
px.area(sex_year, x="anio", y="participaciones", color="sexo", title="Participaciones por sexo").show()